**Residente:** Brener Gomes dos Santos

**Data:** 21/11/2025

In [117]:
import pandas as pd

# Carrega o dataset Iris do arquivo CSV.
# `header=None` indica que o arquivo não tem cabeçalho padrão.
# `skiprows=1` pula a primeira linha do arquivo, que parece ser metadados e não parte dos dados.
# `sep=','` especifica que o delimitador principal é uma vírgula. Isso resultará em 5 colunas,
# onde a última (índice 4) conterá o id da espécie e a informação extra (ex: '0;;Train').
iris_df = pd.read_csv('/content/iris.csv', header=None, skiprows=1, sep=',')

# A última coluna (índice 4) contém o ID da espécie e uma informação extra (;;Train/;;Teste).
# Vamos dividir essa coluna em duas novas colunas.

# Primeiro, garanta que a coluna é string para poder usar .str.split()
last_col_data = iris_df.iloc[:, 4].astype(str)

# Divide a coluna pelo delimitador ';;' em duas partes
split_parts = last_col_data.str.split(';;', expand=True)

# Adiciona as partes divididas ao DataFrame como novas colunas
iris_df['species_id'] = split_parts[0]
iris_df['extra_info'] = split_parts[1]

# Remove a coluna original que continha a informação combinada
iris_df = iris_df.drop(columns=iris_df.columns[4])

# Atribui os nomes corretos às colunas restantes.
iris_df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species', 'extra_info']

# Converte a coluna 'species' para tipo numérico
iris_df['species'] = pd.to_numeric(iris_df['species'])

# Identifica as características numéricas e a variável alvo
X_iris = iris_df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y_iris = iris_df['species']

print("Iris dataset carregado:")
print(f"Número de amostras: {X_iris.shape[0]}")
print(f"Número de atributos (numéricos): {X_iris.shape[1]}")
# Converte os valores únicos da variável alvo para string antes de juntá-los para evitar erros com tipos numéricos.
print(f"Classes de classificação: {y_iris.nunique()} ('{', '.join(y_iris.unique().astype(str).tolist())}')")
print("Primeiras 5 linhas dos atributos (X):")
display(X_iris.head())
print("Primeiras 5 linhas do target (y):")
display(y_iris.head())

Iris dataset carregado:
Número de amostras: 150
Número de atributos (numéricos): 4
Classes de classificação: 3 ('0, 1, 2')
Primeiras 5 linhas dos atributos (X):


,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


Primeiras 5 linhas do target (y):


,species
0,0
1,0
2,0
3,0
4,0


In [118]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score

# Define uma função para avaliar os modelos de classificação.
# A função treina o modelo, faz previsões e calcula a acurácia.
def avaliar_modelo(
    modelo, X_train, X_test, y_train, y_test, average='macro'
):
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    return {
        "accuracy": accuracy
    }

In [119]:
# Divide o dataset Iris em conjuntos de treino e teste.
# Utiliza 70% dos dados para treino e 30% para teste, garantindo a estratificação
# para manter a proporção das classes no treino e no teste.
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

# Define o pré-processador para o dataset Iris.
# Como as features já são numéricas, realiza apenas o escalonamento utilizando StandardScaler.
preprocessor_iris = Pipeline([('scaler', StandardScaler())])

# Inicializa uma lista para armazenar os resultados de todos os modelos.
resultados_iris = []

print("Dataset Iris dividido em treino e teste e pré-processador definido.")

Dataset Iris dividido em treino e teste e pré-processador definido.


In [120]:
# Configura e avalia o modelo de Regressão Logística com penalidade L2.
# Modelo Linear com Norma 2 (Ridge)
# Este é um bom modelo linear para classificação multiclasse.
print("\n=== Treinando e avaliando: LogReg_L2_Iris ===")
model_logreg_l2_iris = Pipeline([
    ("preprocess", preprocessor_iris),
    ("clf", LogisticRegression(penalty="l2", solver="lbfgs", max_iter=2000, multi_class='multinomial', random_state=42))
])

res_logreg_l2_iris = avaliar_modelo(
    model_logreg_l2_iris,
    X_train_iris,
    X_test_iris,
    y_train_iris,
    y_test_iris,
    average="macro"
)
resultados_iris.append({
    "modelo": "LogReg_L2_Iris",
    "accuracy": res_logreg_l2_iris["accuracy"],
})
print("Acurácia:", res_logreg_l2_iris["accuracy"])


=== Treinando e avaliando: LogReg_L2_Iris ===
Acurácia: 0.9111111111111111


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [121]:
# Configura e avalia o modelo de Regressão Logística com penalidade L1.
# Modelo Linear com Norma 1 (Lasso)
# Para L1 com LogisticRegression, utiliza-se o solver 'liblinear' e multi_class='ovr'.
print("\n=== Treinando e avaliando: LogReg_L1_Iris ===")
model_logreg_l1_iris = Pipeline([
    ("preprocess", preprocessor_iris),
    ("clf", LogisticRegression(penalty="l1", solver="liblinear", max_iter=2000, multi_class='ovr', random_state=42))
])

res_logreg_l1_iris = avaliar_modelo(
    model_logreg_l1_iris,
    X_train_iris,
    X_test_iris,
    y_train_iris,
    y_test_iris,
    average="macro"
)
resultados_iris.append({
    "modelo": "LogReg_L1_Iris",
    "accuracy": res_logreg_l1_iris["accuracy"],
})
print("Acurácia: ", res_logreg_l1_iris["accuracy"])


=== Treinando e avaliando: LogReg_L1_Iris ===
Acurácia:  0.8444444444444444


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


In [122]:
# Configura e avalia o modelo de Regressão Logística com penalidade ElasticNet.
# Modelo Linear com Norma 1 e 2 (Elastic Net)
# Para ElasticNet, utiliza-se o solver 'saga' e multi_class='multinomial'.
print("\n=== Treinando e avaliando: LogReg_ElasticNet_Iris ===")
model_logreg_elasticnet_iris = Pipeline([
    ("preprocess", preprocessor_iris),
    ("clf", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.5,
        max_iter=4000,
        multi_class='multinomial',
        random_state=42
    ))
])

res_logreg_elasticnet_iris = avaliar_modelo(
    model_logreg_elasticnet_iris,
    X_train_iris,
    X_test_iris,
    y_train_iris,
    y_test_iris,
    average="macro"
)
resultados_iris.append({
    "modelo": "LogReg_ElasticNet_Iris",
    "accuracy": res_logreg_elasticnet_iris["accuracy"],
})
print("Acurácia: ", res_logreg_elasticnet_iris["accuracy"])


=== Treinando e avaliando: LogReg_ElasticNet_Iris ===
Acurácia:  0.9111111111111111


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [123]:
# Configura e avalia o modelo SGDClassifier com perda logística e penalidade L2.
# Regressão Logística Multinomial / Modelo Linear com Norma 2 (Ridge)
# Este modelo também é uma forma de regressão logística, mas otimizada com Gradiente Descendente Estocástico.
print("\n=== Treinando e avaliando: SGD_logistic_L2_Iris ===")
model_sgd_l2_iris = Pipeline([
    ("preprocess", preprocessor_iris),
    ("clf", SGDClassifier(loss="log_loss", penalty="l2", max_iter=4000, random_state=42, class_weight='balanced'))
])

res_sgd_l2_iris = avaliar_modelo(
    model_sgd_l2_iris,
    X_train_iris,
    X_test_iris,
    y_train_iris,
    y_test_iris,
    average="macro"
)
resultados_iris.append({
    "modelo": "SGD_logistic_L2_Iris",
    "accuracy": res_sgd_l2_iris["accuracy"],
})
print("Acurácia: ", res_sgd_l2_iris["accuracy"])


=== Treinando e avaliando: SGD_logistic_L2_Iris ===
Acurácia:  0.8888888888888888


In [124]:
# Configura e avalia o modelo SGDClassifier com perda logística e penalidade ElasticNet.
# Regressão Logística Multinomial / Modelo Linear com Norma 1 e 2 (Elastic Net)
# Este modelo também usa Gradiente Descendente Estocástico para otimização.
print("\n=== Treinando e avaliando: SGD_logistic_ElasticNet_Iris ===")
model_sgd_elasticnet_iris = Pipeline([
    ("preprocess", preprocessor_iris),
    ("clf", SGDClassifier(loss="log_loss", penalty="elasticnet", l1_ratio=0.5, max_iter=4000, random_state=42, class_weight='balanced'))
])

res_sgd_elasticnet_iris = avaliar_modelo(
    model_sgd_elasticnet_iris,
    X_train_iris,
    X_test_iris,
    y_train_iris,
    y_test_iris,
    average="macro"
)
resultados_iris.append({
    "modelo": "SGD_logistic_ElasticNet_Iris",
    "accuracy": res_sgd_elasticnet_iris["accuracy"],
})
print("Acurácia: ", res_sgd_elasticnet_iris["accuracy"])


=== Treinando e avaliando: SGD_logistic_ElasticNet_Iris ===
Acurácia:  0.8666666666666667


In [125]:
# Cria um DataFrame com todos os resultados obtidos e ordena pela acurácia.
df_resultados_iris = pd.DataFrame(resultados_iris).sort_values(by="accuracy", ascending=False)
print("\nResumo das métricas de acurácia para o Dataset Iris:")
display(df_resultados_iris)


Resumo das métricas de acurácia para o Dataset Iris:


,modelo,accuracy
0,LogReg_L2_Iris,0.911111
2,LogReg_ElasticNet_Iris,0.911111
3,SGD_logistic_L2_Iris,0.888889
4,SGD_logistic_ElasticNet_Iris,0.866667
1,LogReg_L1_Iris,0.844444


In [126]:
import pandas as pd

# 1. Carrega o arquivo 'Titanic-Dataset.csv' em um DataFrame do pandas
titanic_df = pd.read_csv('/content/Titanic-Dataset.csv')

# 2. Exibe as primeiras 5 linhas do titanic_df
print("Primeiras 5 linhas do DataFrame Titanic:")
display(titanic_df.head())

# 3. Verifica as informações gerais do DataFrame
print("\nInformações gerais do DataFrame Titanic:")
titanic_df.info()

# 4. Identifica a presença de valores faltantes em cada coluna
print("\nContagem de valores faltantes por coluna no DataFrame Titanic:")
display(titanic_df.isnull().sum())

Primeiras 5 linhas do DataFrame Titanic:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



Informações gerais do DataFrame Titanic:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

Contagem de valores faltantes por coluna no DataFrame Titanic:


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [127]:
X_titanic = titanic_df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin', 'Survived'])
y_titanic = titanic_df['Survived']

print("Features (X_titanic) shape:", X_titanic.shape)
print("Target (y_titanic) shape:", y_titanic.shape)

Features (X_titanic) shape: (891, 7)
Target (y_titanic) shape: (891,)


In [128]:
numerical_features_titanic = X_titanic.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features_titanic = X_titanic.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numerical features para o dataset Titanic:", numerical_features_titanic)
print("Categorical features para o dataset Titanic:", categorical_features_titanic)

Numerical features para o dataset Titanic: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Categorical features para o dataset Titanic: ['Sex', 'Embarked']


In [129]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# 2. Pipeline para características numéricas
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 3. Pipeline para características categóricas
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# 4. Combina os pipelines numéricos e categóricos em um ColumnTransformer
preprocessor_titanic = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features_titanic),
        ('cat', categorical_transformer, categorical_features_titanic)
    ])

In [130]:
from sklearn.model_selection import train_test_split

X_train_titanic, X_test_titanic, y_train_titanic, y_test_titanic = train_test_split(
    X_titanic, y_titanic, test_size=0.3, random_state=42, stratify=y_titanic
)

print("Dataset Titanic dividido em conjuntos de treino e teste.")
print(f"X_train_titanic shape: {X_train_titanic.shape}")
print(f"X_test_titanic shape: {X_test_titanic.shape}")
print(f"y_train_titanic shape: {y_train_titanic.shape}")
print(f"y_test_titanic shape: {y_test_titanic.shape}")

Dataset Titanic dividido em conjuntos de treino e teste.
X_train_titanic shape: (623, 7)
X_test_titanic shape: (268, 7)
y_train_titanic shape: (623,)
y_test_titanic shape: (268,)


In [131]:
resultados_titanic = []
print("Lista 'resultados_titanic' inicializada.")

Lista 'resultados_titanic' inicializada.


In [132]:
print("\n=== Treinando e avaliando: Regressão Logística (Penalidade L2) ===")
# Modelo Linear com Norma 2 (Ridge)
model_logreg_l2_titanic = Pipeline([
    ("preprocess", preprocessor_titanic),
    ("clf", LogisticRegression(penalty="l2", solver="lbfgs", max_iter=2000, random_state=42))
])

res_logreg_l2_titanic = avaliar_modelo(
    model_logreg_l2_titanic,
    X_train_titanic,
    X_test_titanic,
    y_train_titanic,
    y_test_titanic,
    average="binary"
)
resultados_titanic.append({
    "modelo": "Regressão Logística (L2)",
    "accuracy": res_logreg_l2_titanic["accuracy"]
})
print("Acurácia:", res_logreg_l2_titanic["accuracy"])


=== Treinando e avaliando: Regressão Logística (Penalidade L2) ===
Acurácia: 0.7985074626865671


In [133]:
print("\n=== Treinando e avaliando: Regressão Logística (Penalidade L1) ===")
# Modelo Linear com Norma 1 (Lasso)
model_logreg_l1_titanic = Pipeline([
    ("preprocess", preprocessor_titanic),
    ("clf", LogisticRegression(penalty="l1", solver="liblinear", max_iter=2000, random_state=42))
])

res_logreg_l1_titanic = avaliar_modelo(
    model_logreg_l1_titanic,
    X_train_titanic,
    X_test_titanic,
    y_train_titanic,
    y_test_titanic,
    average="binary"
)
resultados_titanic.append({
    "modelo": "Regressão Logística (L1)",
    "accuracy": res_logreg_l1_titanic["accuracy"]
})
print("Acurácia:", res_logreg_l1_titanic["accuracy"])


=== Treinando e avaliando: Regressão Logística (Penalidade L1) ===
Acurácia: 0.7947761194029851


In [134]:
print("\n=== Treinando e avaliando: Regressão Logística (ElasticNet) ===")
# Modelo Linear com Norma 1 e 2 (Elastic Net)
model_logreg_elasticnet_titanic = Pipeline([
    ("preprocess", preprocessor_titanic),
    ("clf", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.5,
        max_iter=4000,
        random_state=42
    ))
])

res_logreg_elasticnet_titanic = avaliar_modelo(
    model_logreg_elasticnet_titanic,
    X_train_titanic,
    X_test_titanic,
    y_train_titanic,
    y_test_titanic,
    average="binary"
)
resultados_titanic.append({
    "modelo": "Regressão Logística (ElasticNet)",
    "accuracy": res_logreg_elasticnet_titanic["accuracy"]
})
print("Acurácia:", res_logreg_elasticnet_titanic["accuracy"])


=== Treinando e avaliando: Regressão Logística (ElasticNet) ===
Acurácia: 0.7985074626865671


In [135]:
print("\n=== Treinando e avaliando: SGDClassifier (Perda Logística L2) ===")
# Regressão Logística Binária / Modelo Linear com Norma 2 (Ridge)
model_sgd_l2_titanic = Pipeline([
    ("preprocess", preprocessor_titanic),
    ("clf", SGDClassifier(loss="log_loss", penalty="l2", max_iter=4000, random_state=42, class_weight='balanced'))
])

res_sgd_l2_titanic = avaliar_modelo(
    model_sgd_l2_titanic,
    X_train_titanic,
    X_test_titanic,
    y_train_titanic,
    y_test_titanic,
    average="binary"
)
resultados_titanic.append({
    "modelo": "SGDClassifier (L2)",
    "accuracy": res_sgd_l2_titanic["accuracy"]
})
print("Acurácia:", res_sgd_l2_titanic["accuracy"])


=== Treinando e avaliando: SGDClassifier (Perda Logística L2) ===
Acurácia: 0.7947761194029851


In [136]:
print("\n=== Treinando e avaliando: SGDClassifier (Perda Logística ElasticNet) ===")
# Regressão Logística Binária / Modelo Linear com Norma 1 e 2 (Elastic Net)
model_sgd_elasticnet_titanic = Pipeline([
    ("preprocess", preprocessor_titanic),
    ("clf", SGDClassifier(loss="log_loss", penalty="elasticnet", l1_ratio=0.5, max_iter=4000, random_state=42, class_weight='balanced'))
])

res_sgd_elasticnet_titanic = avaliar_modelo(
    model_sgd_elasticnet_titanic,
    X_train_titanic,
    X_test_titanic,
    y_train_titanic,
    y_test_titanic,
    average="binary"
)
resultados_titanic.append({
    "modelo": "SGDClassifier (ElasticNet)",
    "accuracy": res_sgd_elasticnet_titanic["accuracy"]
})
print("Acurácia:", res_sgd_elasticnet_titanic["accuracy"])


=== Treinando e avaliando: SGDClassifier (Perda Logística ElasticNet) ===
Acurácia: 0.7835820895522388


In [137]:
df_resultados_titanic = pd.DataFrame(resultados_titanic).sort_values(by="accuracy", ascending=False)
print("\nResumo das métricas de acurácia para o Dataset Titanic:")
display(df_resultados_titanic)


Resumo das métricas de acurácia para o Dataset Titanic:


,modelo,accuracy
0,Regressão Logística (L2),0.798507
2,Regressão Logística (ElasticNet),0.798507
1,Regressão Logística (L1),0.794776
3,SGDClassifier (L2),0.794776
4,SGDClassifier (ElasticNet),0.783582
